# Nepal Script Text Recognition From Ancient Artifacts: Challenges and Opportunities
### Authors:
#### Swornim Nakarmi, Sarin Sthapit, Sahil Ratna Tuladhar, Arya Shakya, Bal Krishna Bal, Rajani Chulyadyo

---

This notebook covers the full development and training pipeline for the text recognition system, including dataset preparation, model architecture design, training, and evaluation.

The notebook is organized into the following sections:
1. **Dataset Preparation**: loading, preprocessing, and splitting the training data
2. **Text Recognition Model**: CRNN architecture trained with CTC loss for sequence-to-sequence transcription
3. **Training**: training loop, loss monitoring, and checkpointing
4. **Evaluation**: character error rate (CER) and qualitative results
5. **Export**: exporting trained models to .h5 format for inference

**Dataset:** [Nepal Script Stone Inscription Dataset](https://www.kaggle.com/dsv/15121632)

## Install and Load Required Packages

In [ ]:
import os
import cv2
import time
import math
import numpy as np
import pandas as pd
import supervision as sv
from jiwer import cer
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, TimeDistributed

from keras.preprocessing import sequence
from sklearn.model_selection import train_test_split

## Load and Prepare Dataset

In [ ]:
data_dir = '<dir_to_dataset>'

charset = pd.read_csv(f'{data_dir}/charset.csv')
characters = charset['Nepal'].values.tolist()

In [ ]:
data = pd.read_csv(f'{data_dir}/transcription.csv')
data = data.sample(frac=1)

image_paths = data['image'].tolist()
texts = data['text'].tolist()

images = []
labels = []
for image, label in zip(image_paths, texts):
    if len(label) <= 100 and label != '':
        images.append(f"{data_dir}/images/{image}.jpg")
        labels.append(label)

In [ ]:
def encode_text(text):
    """
    Encode a unicode string into a sequence of character indices.
    Characters not present in the charset are skipped.

    Args:
        text : Raw unicode string to encode.
    Returns:
        1D numpy array of integer indices into characters.
    """
    return np.array(
        [characters.index(ord(ch)) for ch in text if ord(ch) in characters],
        dtype=np.int32
    )


def decode_text(encoded):
    """
    Decode a sequence of character indices back into a unicode string.
    Stops at the CTC blank token (index == len(characters)).

    Args:
        encoded : 1D numpy array of integer character indices.
    Returns:
        Decoded unicode string.
    """
    chars = []
    for idx in encoded.astype(int):
        if idx == len(characters):    # CTC blank token
            break
        chars.append(chr(characters[idx]))
    
    return ''.join(chars)

In [ ]:
X = np.array(images)

y = [encode_text(label) for label in labels]
y = sequence.pad_sequences(y, value=float(len(characters)), dtype='float32', padding='post')
y = np.array(y, dtype=np.float32)

In [ ]:
IMG_WIDTH  = 508
IMG_HEIGHT = 64


def preprocess_image(img_path):
    """
    Load and preprocess a single image.

    Reads the image from disk, resizes it to (IMG_WIDTH, IMG_HEIGHT) using
    letterboxing to preserve aspect ratio, extracts the grayscale channel,
    transposes to (IMG_WIDTH, IMG_HEIGHT), and normalizes to [0, 1].

    Args:
        img_path : Image file path.
    Returns:
        Preprocessed image as a float32 numpy array of shape (IMG_WIDTH, IMG_HEIGHT).
    """
    img = cv2.imread(img_path.decode('utf-8'))
    img = sv.letterbox_image(image=img, resolution_wh=(IMG_WIDTH, IMG_HEIGHT), color=sv.Color.BLACK)
    img = img[:, :, 0]                  # Extract grayscale channel
    img = cv2.transpose(img)            # (IMG_HEIGHT, IMG_WIDTH) -> (IMG_WIDTH, IMG_HEIGHT)
    
    return img.astype(np.float32) / 255.0


def tf_preprocess(img_path, label):
    """
    TensorFlow-compatible preprocessing wrapper for use in a tf.data pipeline.

    Wraps 'preprocess_image' with tf.numpy_function to allow numpy operations
    inside the pipeline.

    Args:
        img_path : Scalar string tensor containing the image file path.
        label    : Encoded label tensor for the corresponding image.
    Returns:
        Dict with keys 'image' (float32 tensor of shape (IMG_WIDTH, IMG_HEIGHT))
        and 'label' (unchanged label tensor).
    """
    img = tf.numpy_function(preprocess_image, [img_path], tf.float32)
    img.set_shape([IMG_WIDTH, IMG_HEIGHT])
    
    return {'image': img, 'label': label}

In [ ]:
X_train, X_other, y_train, y_other = train_test_split(X, y, test_size=0.3, random_state=33)
X_valid, X_test, y_valid, y_test = train_test_split(X_other, y_other, test_size=0.33, random_state=33)

batch_size = 32

train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(1000)
    .map(tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    tf.data.Dataset.from_tensor_slices((X_valid, y_valid))
    .map(tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .map(tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
def preview_dataset(dataset):
    """
    Preview a single sample from a tf.data dataset.
    Displays the image in both its stored orientation (IMG_WIDTH, IMG_HEIGHT)
    and its original orientation (IMG_HEIGHT, IMG_WIDTH) side by side,
    along with the decoded label and shape info.

    Args:
        dataset : A tf.data.Dataset yielding {'image': ..., 'label': ...} dicts.
    """
    sample = next(iter(dataset.unbatch().take(1)))

    img   = sample['image'].numpy()
    label = decode_text(sample['label'].numpy())

    print(f'Image shape: {img.reshape(1, IMG_WIDTH, IMG_HEIGHT, 1).shape}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))

    axes[0].imshow(img,   cmap='gray')
    axes[0].set_title(f'({IMG_WIDTH}, {IMG_HEIGHT})')
    axes[0].axis('off')

    axes[1].imshow(img.T, cmap='gray')
    axes[1].set_title(f'({IMG_HEIGHT}, {IMG_WIDTH})')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
    
    print(label)

In [ ]:
preview_dataset(train_dataset)

## Build OCR Model

In [ ]:
class CTCLoss(layers.Layer):
    """
    Custom layer that computes CTC loss during training.

    Wraps 'keras.backend.ctc_batch_cost' and registers the loss via
    'add_loss' so it integrates with 'model.compile' and 'model.fit'.
    The layer passes 'y_pred' through unchanged so it can be used as a model output.

    Args:
        name : Optional name for the layer.
    """

    def __init__(self, name = None):
        super().__init__(name=name)
        self.loss_fn = keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred):
        """
        Compute mean CTC loss over the batch and register it.

        Args:
            y_true : Ground truth label tensor of shape (batch, label_length).
            y_pred : Predicted softmax tensor of shape (batch, time_steps, num_classes).
        Returns:
            y_pred unchanged, allowing this layer to be inline in the model graph.
        """
        batch_len = tf.shape(y_true)[0]
        input_length = tf.fill([batch_len, 1], tf.shape(y_pred)[1])
        label_length = tf.fill([batch_len, 1], tf.shape(y_true)[1])

        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(tf.reduce_mean(loss))
        
        return y_pred


def ctc_decode(pred):
    """
    Greedily decode a single CTC prediction into a unicode string.

    Uses 'keras.backend.ctc_decode' with greedy search, then converts
    the resulting index sequence back to text with 'decode_text'.

    Args:
        pred : Softmax prediction array of shape (1, time_steps, num_classes).
    Returns:
        Decoded and stripped unicode string.
    """
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    results   = keras.backend.ctc_decode(pred, input_length=input_len, greedy=True)[0][0][:]
    
    return decode_text(results[0].numpy()).strip()

In [ ]:
def build_conv_block(x, filters, name):
    """
    Build a single Conv2D + BatchNormalization block.

    Args:
        x       : Input tensor.
        filters : Number of convolutional filters.
        name    : Name for the Conv2D layer.
    Returns:
        Output tensor after Conv2D and BatchNormalization.
    """
    x = layers.Conv2D(
        filters, (2, 2), activation='relu', padding='same',
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(1e-4),
        name=name
    )(x)
    
    return layers.BatchNormalization()(x)


def build_ocr_model(num_characters):
    """
    Build and compile the Nepal Script OCR model.

    Architecture:
        - CNN         : 5 Conv+BN blocks with 2 MaxPooling layers for feature extraction.
        - Bridge      : Reshape -> Dense -> BN -> Dropout to project CNN features into sequences.
        - BiLSTM head : 3 stacked Bidirectional LSTM layers for sequence modelling.
        - Output head : TimeDistributed Dense -> BN -> Dropout -> Softmax over (num_characters + 1).
        - Loss        : CTCLoss layer for Connectionist Temporal Classification.

    Args:
        num_characters : Number of characters in the character-set.
    Returns:
        Compiled Keras model with Adam optimizer.
    """
    # Inputs
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name='image', dtype='float32')
    labels = layers.Input(shape=(None,), name='label', dtype='float32')

    # CNN Backbone
    x = build_conv_block(input_img, 32, 'conv_1')
    x = build_conv_block(x, 32, 'conv_2')
    x = build_conv_block(x, 64, 'conv_3')
    x = layers.MaxPooling2D((2, 2), name='max_pool_1')(x)

    x = build_conv_block(x, 64, 'conv_4')
    x = build_conv_block(x, 128, 'conv_5')
    x = layers.MaxPooling2D((2, 2), name='max_pool_2')(x)

    # CNN > Sequence
    x = layers.Reshape(target_shape=(-1, 2048), name='reshape')(x)
    x = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4), name='dense_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # BiLSTM Sequence Modeling
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3), name='blstm_1')(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.3), name='blstm_2')(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.3), name='blstm_3')(x)

    # Output Head
    x = layers.TimeDistributed(
            layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
            name='time_dense')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(
            layers.Dense(num_characters + 1, activation='softmax'),
            name='output')(x)

    output = CTCLoss(name='ctc_loss')(labels, x)

    # Compile
    model = keras.Model(
        inputs=[input_img, labels],
        outputs=output,
        name='nepal_script_ocr_model'
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
    model.summary()
    
    return model

In [ ]:
model = build_ocr_model(num_characters=len(characters))

## Train Model

In [ ]:
epochs = 250
n_batches = math.ceil(train_dataset.cardinality().numpy())
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

checkpoint = keras.callbacks.ModelCheckpoint(
    filepath='ocr_model_{epoch:02d}.keras',
    save_freq=20 * n_batches,
    save_best_only=False,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs,
    callbacks=[checkpoint, lr_scheduler]
)

end_time = time.time()

elapsed = end_time - start_time
print(f'Training took {elapsed:.1f}s ({elapsed/60:.1f} min)')

## Plot Loss Curves

In [ ]:
def plot_loss_curves(history, save_path = 'training_loss_curve.png'):
    """
    Plot and save training and validation CTC loss curves.

    Args:
        history   : History object.
        save_path : File path to save the plot image.
    """
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(loss) + 1)

    plt.figure(figsize=(12, 6), dpi=100)
    plt.rcParams.update({'font.size': 20})

    plt.plot(epochs, loss, linewidth=2, label='Train Loss')
    plt.plot(epochs, val_loss, linewidth=2, label='Val Loss')

    plt.xlabel('Epoch', fontsize=20)
    plt.ylabel('CTC Loss', fontsize=20)
    plt.title('Loss Curve', fontsize=20)
    plt.legend(prop={'size': 20})
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    
    print(f'Loss curve saved to {save_path}')

In [ ]:
plot_loss_curves(history, save_path='training_loss_curve.png')

## Get Prediction Model from the Trained Model

In [ ]:
prediction_model = keras.Model(
    inputs=model.get_layer(name='image').output,
    outputs=model.get_layer(name='output').output
)

## Validation Sample Predictions

In [ ]:
for i, sample in enumerate(validation_dataset.unbatch().take(10)):
    img  = sample['image'].numpy()     # (508, 64)
    label = decode_text(sample['label'].numpy().astype(int))

    plt.figure()
    plt.imshow(img.T, cmap='gray')     # transpose back to (64, 508)
    plt.axis('off')
    plt.show()

    pred = prediction_model.predict(img.reshape(1, 508, 64, 1), verbose=0)
    print(f'Ground truth: {label}')
    print(f'Predicted   : {ctc_decode(pred)}')
    print('-' * 40)

## Model Evaluation

In [ ]:
def evaluate(model, dataset):
    """
    Evaluate the OCR model performance using CER metric.

    Runs greedy CTC decoding on each sample, compares against ground truth,
    and reports mean Character Error Rate. Samples where either the ground
    truth or prediction is empty are skipped.

    Args:
        model   : Trained prediction model (without CTC loss layer).
        dataset : A tf.data.Dataset yielding {'image': ..., 'label': ...} dicts.
    Returns:
        'mean_cer' as plain Python floats (percentage).
    """
    cers = []

    for sample in dataset.unbatch():
        img = sample['image'].numpy().reshape(1, IMG_WIDTH, IMG_HEIGHT, 1)
        label = decode_text(sample['label'].numpy().astype(int))
        predicted = ctc_decode(model.predict(img, verbose=0))

        if not label or not predicted:
            continue

        cers.append(cer(label, predicted))

    mean_cer = float(np.mean(cers) * 100)

    print(f'Samples evaluated : {len(cers)}')
    print(f'CER               : {mean_cer:.2f}%')

    return mean_cer

In [ ]:
# train dataset
evaluate(prediction_model, train_dataset)

In [ ]:
# validation dataset
evaluate(prediction_model, validation_dataset)

In [ ]:
# test dataset
evaluate(prediction_model, test_dataset)

## Save Model

In [ ]:
prediction_model.save('ocr_model.keras')